# OSRT Ostinato — v7 pretrain on Colab

Target runtime: **RTX PRO 6000 Blackwell, 96GB**.

Run the cells in order. Cell 1 is a gate — if it does not report `sm_120`,
stop and read its verdict before spending anything.

**Sessions are capped and the VM's disk is wiped on release.** Cell 5 sets
`--hf-repo`, which pulls the newest checkpoint before training and pushes
each new one as it is written. Without it a disconnect loses the run.

Do **not** background the training cell (`nohup`, `&`). Colab tears down the
runtime when the foreground cell returns, so a backgrounded trainer dies
with it — run it in the foreground and leave the tab open.


## 1 · Gate: what card is this, and can the expert path go low-precision?

This is also the first half of roadmap gate **G7**. Routed experts are ~84%
of v7's parameters and run through the private `torch._grouped_mm`; if that
refuses FP8, the NVFP4 case in §13.3 covers under a third of the model.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv


## 2 · Install


In [ ]:
!git clone -q https://github.com/CodeHalwell/OSRT-Ostinato.git
%cd OSRT-Ostinato
!pip install -q -e . 2>&1 | tail -2
!PYTHONPATH=src python scripts/probe_gpu.py


## 3 · Secrets

Add `HF_TOKEN` (write access to your checkpoint repo) and `WANDB_API_KEY`
in the Colab **Secrets** panel (🔑, left sidebar). They never touch this
notebook or git.


In [ ]:
import os

from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert os.environ['HF_TOKEN'] and os.environ['WANDB_API_KEY'], 'set both in Secrets'
print('secrets set')


## 4 · Build the tokenizer and confirm the shape

`compute_budget.py` is the only trusted source for parameter counts — the
repo deliberately states none in any name. Expect **968,468,355 physical /
263,035,779 active**; anything else means the config drifted.


In [ ]:
!python scripts/build_tokenizer_v7.py --out tokenizer
!PYTHONPATH=src python scripts/compute_budget.py


## 5 · The run

This is the trunk, not a sample. `PretrainConfig` holds the budget: 18,000 steps
≈ 5.31B tokens, ~1× Chinchilla on active params (roadmap §14.8 — the bet).

Set `HF_CKPT_REPO` to a **private** repo you own. Every checkpoint is pushed
there as it is written and the newest is pulled before training, so when
this session dies — it will — re-running this cell continues the same run.
Nothing else is needed for resume.

Leave the tab open. Do **not** background the cell.


> **Card size.** `PretrainConfig` is sized for a 192 GB B200 (32K tokens per micro-batch,
> roadmap §13b). On the 96 GB RTX PRO 6000 pass `--micro-batch-scale 0.5` to halve every
> phase's micro-batch and double its accumulation — same tokens per step, same run.


In [ ]:
HF_CKPT_REPO = 'HallD/osrt-v7-ckpt'   # <-- your PRIVATE repo

!PYTHONPATH=src python -m osrt.train_main \
    --tokenizer-path ./tokenizer \
    --ckpt-dir ./checkpoints/v7 \
    --hf-repo {HF_CKPT_REPO} \
    --wandb-run-name osrt-v7-trunk


## 6 · What to watch

The early-stop criteria will end the run on router or loop collapse — that
is the design. If it stops, the log names the criterion. Otherwise:

| signal | healthy | worry |
|---|---|---|
| `loss` | falling, no unrecovered spikes | flat, or spiking |
| `moe/dead_experts_total` | 0 | > 0 — at E=28 each is 3.6% of a block |
| `moe/b*/loop*/load_entropy` | near ln(28) ≈ 3.33 | falling — router collapse |
| `loop/update_norm_l*` | non-trivial at every loop | last loops → 0 = loop collapse |
| `loop_hidden_norm_ratio` | ≈ 1, flat | rising — residual explosion (§17.3) |
| `moe/b*/bias_loop_spread` | flat | rising — routing diverging across loops |
| `muon/ortho_err` | small, flat | rising — Newton–Schulz not converging |

The bets this run embodies, and what would falsify each, are in roadmap §19.
